In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# **Mlflow and dagshub initialisation**

In [2]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import dagshub
import mlflow

dagshub.init(repo_owner="luchit22", repo_name="ML-fraud-detection",mlflow=True)
mlflow.set_experiment("random_forest")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=19f28a81-97c3-4128-addb-4328bf204b23&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f6fd2fd938efea8a5dbaa1f8607e3b258f6cbeb70dce54e0943d47c1ea267abd




Accessing as luchit22

Initialized MLflow to track repo "luchit22/ML-fraud-detection"

Repository luchit22/ML-fraud-detection initialized!

2026/05/02 15:52:22 INFO mlflow.tracking.fluent: Experiment with name 'random_forest' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3245794c6c1940c59f2745609007023f', creation_time=1777737142442, experiment_id='1', last_update_time=1777737142442, lifecycle_stage='active', name='random_forest', tags={}, trace_location=None, workspace='default'>

In [4]:
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# **data cleaning**

In [5]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

train_transaction.columns = train_transaction.columns.str.replace("-", "_")
train_identity.columns = train_identity.columns.str.replace("-", "_")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)

(590540, 434)


In [6]:
y = train["isFraud"]
X = train.drop(columns=["isFraud"])

In [7]:
missing_ratio = X.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.90].index.tolist()

X = X.drop(columns=cols_to_drop)

print("Dropped columns with >90% missing:", len(cols_to_drop))

Dropped columns with >90% missing: 12


In [8]:
X = X.select_dtypes(exclude=["object"])

print("Numeric shape:", X.shape)

Numeric shape: (590540, 392)


In [9]:
X = X.fillna(-999)

# **feature engineering**

In [10]:
X_fe = train.drop(columns=["isFraud"])
X_fe = X_fe.drop(columns=cols_to_drop)

X["TransactionAmt_log"] = np.log1p(X_fe["TransactionAmt"])
X["TransactionDT_days"] = X_fe["TransactionDT"] / (3600 * 24)
X["TransactionDT_hours"] = X_fe["TransactionDT"] / 3600
X["num_missing_original"] = X_fe.isnull().sum(axis=1)

X = X.fillna(-999)

print("Shape after feature engineering:", X.shape)

Shape after feature engineering: (590540, 396)


In [11]:
X_sample, _, y_sample, _ = train_test_split(
    X,
    y,
    train_size=100000,
    stratify=y,
    random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

In [12]:
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Fraud ratio train:", y_train.mean())
print("Fraud ratio valid:", y_valid.mean())

Train: (80000, 396)
Valid: (20000, 396)
Fraud ratio train: 0.0349875
Fraud ratio valid: 0.035


# **training without selectoin**

In [13]:
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

In [14]:
rf_baseline.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_jobs=-1,
                       random_state=42)

In [15]:
train_proba = rf_baseline.predict_proba(X_train)[:, 1]
valid_proba = rf_baseline.predict_proba(X_valid)[:, 1]

train_auc = roc_auc_score(y_train, train_proba)
valid_auc = roc_auc_score(y_valid, valid_proba)

print("Train ROC-AUC:", train_auc)
print("Validation ROC-AUC:", valid_auc)
print("AUC gap:", train_auc - valid_auc)

Train ROC-AUC: 0.924862609654982
Validation ROC-AUC: 0.877070133234641
AUC gap: 0.04779247642034101


In [16]:
with mlflow.start_run(run_name="RandomForest_Baseline"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("train_size", 100000)
    mlflow.log_param("missing_strategy", "fillna_-999")
    mlflow.log_param("used_numeric_only", True)
    mlflow.log_param("dropped_missing_columns_threshold", 0.90)
    mlflow.log_param("num_features", X_train.shape[1])

    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("valid_auc", valid_auc)
    mlflow.log_metric("auc_gap", train_auc - valid_auc)

    mlflow.sklearn.log_model(rf_baseline, artifact_path="random_forest_baseline")

2026/05/02 16:11:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 16:11:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_Baseline at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1/runs/b7e9cbb7323b44449a6188bcb163a08b
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1


# **feature selection**

In [17]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_baseline.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(30)

,feature,importance
301,V264,0.025230
22,C13,0.022607
128,V91,0.021913
23,C14,0.021855
19,C10,0.020933
302,V265,0.019016
331,V294,0.017797
107,V70,0.017557
13,C4,0.017293
127,V90,0.017223


In [18]:
top_features = feature_importance.head(100)["feature"].tolist()

X_train_top = X_train[top_features]
X_valid_top = X_valid[top_features]

print("Selected top features:", len(top_features))

Selected top features: 100


# **training**

In [19]:
rf_selected = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

In [20]:
rf_selected.fit(X_train_top, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=5, n_estimators=150, n_jobs=-1,
                       random_state=42)

In [21]:
train_proba_selected = rf_selected.predict_proba(X_train_top)[:, 1]
valid_proba_selected = rf_selected.predict_proba(X_valid_top)[:, 1]

train_auc_selected = roc_auc_score(y_train, train_proba_selected)
valid_auc_selected = roc_auc_score(y_valid, valid_proba_selected)

print("Train ROC-AUC:", train_auc_selected)
print("Validation ROC-AUC:", valid_auc_selected)
print("AUC gap:", train_auc_selected - valid_auc_selected)

Train ROC-AUC: 0.9581561356154974
Validation ROC-AUC: 0.8842512953367876
AUC gap: 0.0739048402787098


In [22]:
with mlflow.start_run(run_name="RandomForest_FeatureImportance_Top100"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("feature_selection", "feature_importance_top100")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("train_size", 100000)
    mlflow.log_param("num_selected_features", len(top_features))

    mlflow.log_metric("train_auc", train_auc_selected)
    mlflow.log_metric("valid_auc", valid_auc_selected)
    mlflow.log_metric("auc_gap", train_auc_selected - valid_auc_selected)

    mlflow.sklearn.log_model(rf_selected, artifact_path="random_forest_top100")

2026/05/02 16:18:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 16:18:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_FeatureImportance_Top100 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1/runs/93797a98ca4b432da0ed85881b246d06
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1


In [23]:
rf_configs = [
    {
        "name": "RF_depth8_leaf10",
        "n_estimators": 100,
        "max_depth": 8,
        "min_samples_leaf": 10
    },
    {
        "name": "RF_depth12_leaf5",
        "n_estimators": 150,
        "max_depth": 12,
        "min_samples_leaf": 5
    },
    {
        "name": "RF_depth16_leaf3",
        "n_estimators": 200,
        "max_depth": 16,
        "min_samples_leaf": 3
    }
]

In [24]:
rf_results = []

for config in rf_configs:
    with mlflow.start_run(run_name=config["name"]):
        model = RandomForestClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            min_samples_leaf=config["min_samples_leaf"],
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        )

        model.fit(X_train_top, y_train)

        train_proba = model.predict_proba(X_train_top)[:, 1]
        valid_proba = model.predict_proba(X_valid_top)[:, 1]

        train_auc = roc_auc_score(y_train, train_proba)
        valid_auc = roc_auc_score(y_valid, valid_proba)
        auc_gap = train_auc - valid_auc

        mlflow.log_param("model", "RandomForest")
        mlflow.log_param("feature_selection", "feature_importance_top100")
        mlflow.log_params(config)

        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("valid_auc", valid_auc)
        mlflow.log_metric("auc_gap", auc_gap)

        rf_results.append({
            **config,
            "train_auc": train_auc,
            "valid_auc": valid_auc,
            "auc_gap": auc_gap
        })

rf_results_df = pd.DataFrame(rf_results).sort_values("valid_auc", ascending=False)
rf_results_df

🏃 View run RF_depth8_leaf10 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1/runs/3d7bd7082411487fb12c5778f4bd1865
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1
🏃 View run RF_depth12_leaf5 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1/runs/6eb2ae88cefd486cbc57ab16a71d69b5
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1
🏃 View run RF_depth16_leaf3 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1/runs/2131c277b1964da18a97f64e856b0428
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1


,name,n_estimators,max_depth,min_samples_leaf,train_auc,valid_auc,auc_gap
2,RF_depth16_leaf3,200,16,3,0.990195,0.884385,0.105811
1,RF_depth12_leaf5,150,12,5,0.958156,0.884251,0.073905
0,RF_depth8_leaf10,100,8,10,0.900609,0.871361,0.029248


In [28]:
best_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

best_rf.fit(X_train, y_train)

KeyboardInterrupt: 

In [26]:
train_proba = best_rf.predict_proba(X_train)[:, 1]
valid_proba = best_rf.predict_proba(X_valid)[:, 1]

train_auc = roc_auc_score(y_train, train_proba)
valid_auc = roc_auc_score(y_valid, valid_proba)
auc_gap = train_auc - valid_auc

print("Train:", train_auc)
print("Valid:", valid_auc)

Train: 0.9488460635454008
Valid: 0.8841560325684679


In [32]:
with mlflow.start_run(run_name="RandomForest_BEST_Model"):

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_param("train_size", len(X_train_top))
    mlflow.log_param("num_features", X_train_top.shape[1])
    mlflow.log_param("feature_selection", "feature_importance_top100")
    mlflow.log_param("missing_strategy", "fillna_-999")
    mlflow.log_param("used_numeric_only", True)

    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("valid_auc", valid_auc)
    mlflow.log_metric("auc_gap", auc_gap)

    mlflow.sklearn.log_model(
        best_rf,
        artifact_path="random_forest_best_model"
    )

2026/05/02 16:43:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 16:44:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_BEST_Model at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1/runs/6df07fe7076d4dd78d6439e7e8b83882
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/1
